<a href="https://colab.research.google.com/github/ynam0327-afk/REDRED/blob/main/smishing_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!rm -rf /content/REDRED
!git clone https://github.com/ynam0327-afk/REDRED.git
%cd /content/REDRED

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: could not create work tree dir 'REDRED': No such file or directory
[Errno 2] No such file or directory: '/content/REDRED'
/content/REDRED


In [19]:
%cd /content
!rm -rf REDRED
!git clone https://github.com/ynam0327-afk/REDRED.git
%cd REDRED

!jupyter nbconvert --to script official_sms_check.ipynb
!jupyter nbconvert --to script text_match.ipynb

import os, json

def to_py(name):
    txt_path = f"{name}.txt"
    py_path = f"{name}.py"
    if os.path.exists(txt_path) and not os.path.exists(py_path):
        os.rename(txt_path, py_path)

to_py("official_sms_check")
to_py("text_match")

def repair_if_notebook_json(py_path):
    if not os.path.exists(py_path):
        print(f"[없음] {py_path}")
        return
    with open(py_path, encoding="utf-8") as f:
        content = f.read()
    try:
        nb = json.loads(content)
    except json.JSONDecodeError:
        return
    if "cells" not in nb:
        return
    code = "\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
    with open(py_path, "w", encoding="utf-8") as f:
        f.write(code)
    print(f"{py_path} 복구 완료 (JSON -> 실제 코드)")

for name in ["content_authenticity", "cross_validation", "model", "region_extractor", "text_match", "official_sms_check"]:
    repair_if_notebook_json(f"{name}.py")

print(">>> 여기까지는 항상 도달했음")   # ← 지금까지 로그가 항상 여기서 멈춤

import pandas as pd
from datetime import date as date_cls
from urllib.parse import urlparse

print(">>> content_authenticity import 시작")
from content_authenticity import content_authenticity_score
print(">>> content_authenticity 완료")

print(">>> text_match import 시작")
from text_match import match_sms_to_call119
print(">>> text_match 완료")

print(">>> model import 시작 (여기서 RF/XGBoost 학습이 돕니다 - 시간 걸림)")
from model import url_risk_score_model, is_official_domain, load_url_model
print(">>> model 완료")

print(">>> official_sms_check import 시작")
from official_sms_check import official_sms_reliability
print(">>> official_sms_check 완료")

print(">>> cross_validation import 시작")
from cross_validation import SourceMatch, combined_disaster_reliability
print(">>> cross_validation 완료")

print(">>> region_extractor import 시작")
from region_extractor import extract_region_from_text
print(">>> 전부 완료!")

/content
Cloning into 'REDRED'...
remote: Enumerating objects: 255, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 255 (delta 41), reused 18 (delta 18), pack-reused 190 (from 2)
Receiving objects: 100% (255/255), 252.98 KiB | 5.88 MiB/s, done.
Resolving deltas: 100% (83/83), done.
/content/REDRED
[NbConvertApp] WARNING | pattern 'official_sms_check.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.sh

In [20]:
%cd /content
!rm -rf REDRED
!git clone https://github.com/ynam0327-afk/REDRED.git
%cd REDRED

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from datetime import date as date_cls
from urllib.parse import urlparse
import os

# .ipynb -> .py 변환 (text_match도 빠뜨리지 말고 같이)
!jupyter nbconvert --to script official_sms_check.ipynb
!jupyter nbconvert --to script text_match.ipynb

for name in ["official_sms_check", "text_match"]:
    txt_path = f"{name}.txt"
    py_path = f"{name}.py"
    if os.path.exists(txt_path) and not os.path.exists(py_path):
        os.rename(txt_path, py_path)

from content_authenticity import content_authenticity_score
from text_match import match_sms_to_call119
from model import url_risk_score_model, is_official_domain, load_url_model
from official_sms_check import official_sms_reliability
from cross_validation import SourceMatch, combined_disaster_reliability
from region_extractor import extract_region_from_text

DATA_DIR = "/content/drive/MyDrive/REDRED"

_RF_MODEL = load_url_model(f"{DATA_DIR}/rf_url_model.joblib")

_FIRE_DB = pd.read_csv(f"{DATA_DIR}/fire_events_region_normalized.csv")
_FIRE_DB["report_date"] = _FIRE_DB["report_date"].astype(str)

_CALL119_BY_CITY = {
    "서울특별시": pd.read_parquet(f"{DATA_DIR}/seoul_119_2024_dates.parquet"),
    "부산광역시": pd.read_csv(f"{DATA_DIR}/busan_119_2024_dates.csv"),
}
for _df in _CALL119_BY_CITY.values():
    _df["dclr_ymd"] = _df["dclr_ymd"].astype(str)

# (여기부터 _check_fire_db, _check_call119, _extract_domain, process_message 함수 정의는 그대로)
# ---------------------------------------------------------------------------
# 개별 소스 조회 함수
# ---------------------------------------------------------------------------

def _check_fire_db(region: str, report_date: str) -> SourceMatch:
    if not region:
        return SourceMatch(False)
    sido = region.split()[0] if region.split() else None
    cand = _FIRE_DB[(_FIRE_DB["report_date"] == report_date) & (_FIRE_DB["sido_official"] == sido)]
    if len(cand) == 0:
        return SourceMatch(False)
    return SourceMatch(True, confidence=0.6)


def _check_call119(message: str, region: str, sms_date: str) -> SourceMatch:
    sido = region.split()[0] if region and region.split() else None
    call119_df = _CALL119_BY_CITY.get(sido)
    if call119_df is None:
        return None  # 서울/부산 외 지역 - 애초에 커버리지 없음(정보부족 처리 대상)

    row = pd.Series({"date": sms_date, "region": region, "message": message,
                      "disaster_type": None})
    result = match_sms_to_call119(row, call119_df)
    if result is None:
        return SourceMatch(False)

    hour_diff = result.get("hour_diff")
    confidence = max(0.0, 1 - hour_diff / 12) if hour_diff is not None else 0.5
    return SourceMatch(True, confidence=confidence)


def _extract_domain(url: str) -> str:
    if not url:
        return ""
    if not url.startswith(("http://", "https://")):
        url = "http://" + url
    return urlparse(url).netloc.lower()


# ---------------------------------------------------------------------------
# 외부(ingest-worker 등)에서 부르는 단일 진입점
# ---------------------------------------------------------------------------

def process_message(raw_text: str, url: str | None, region: str | None = None,
                     sms_date: str | None = None,
                     official_service_key: str | None = None) -> dict:

    region_note = None
    if region is None:
        region_result = extract_region_from_text(raw_text)
        region = region_result["region_string"]
        region_note = region_result["note"]

    sms_date = sms_date or date_cls.today().isoformat()
    ymd = sms_date.replace("-", "")

    # 1) URL 위험도 - model.py의 RF 모델 사용 (domain_module의 규칙 기반은 안 씀)
    domain = _extract_domain(url)
    is_wl = bool(url) and is_official_domain(domain)
    url_risk_score = 0.0 if (not url or is_wl) else url_risk_score_model(url, _RF_MODEL)

    # 2) 재난정보 신뢰도 - 공식API > (소방청DB + 119신고접수 결합) > 콘텐츠 판단 순
    official_score = None
    if official_service_key:
        try:
            official_result = official_sms_reliability(raw_text, ymd, region)
            official_score = official_result.get("score")
        except Exception:
            official_score = None  # API 장애 시에도 파이프라인은 계속 진행

    fire_db_match = _check_fire_db(region, sms_date)
    call119_match = _check_call119(raw_text, region, sms_date)

    disaster = combined_disaster_reliability(
        fire_db_match, call119_match, raw_text, official_score
    )

    return {
        "url_risk_score": round(url_risk_score, 4),
        "text_authenticity_score": disaster["score"],
        "detail": {
            "url_whitelisted": is_wl,
            "region_used": region,
            "region_extraction_note": region_note,
            "disaster_matched_sources": disaster["matched_sources"],
            "disaster_note": disaster["note"],
        },
    }


# ---------------------------------------------------------------------------
# 검증
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    # region을 넘기지 않음 - 원문에서 자동 추출되는지 확인
    result = process_message(
        raw_text="06:40 욱성화학 화재 관련하여 화재현장에 폭발위험은 전혀 없습니다 [금정구]",
        url=None,
        sms_date="2024-08-01",
    )
    print(result)


/content
Cloning into 'REDRED'...
remote: Enumerating objects: 255, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 255 (delta 41), reused 18 (delta 18), pack-reused 190 (from 2)
Receiving objects: 100% (255/255), 252.98 KiB | 5.06 MiB/s, done.
Resolving deltas: 100% (83/83), done.
/content/REDRED
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[NbConvertApp] WARNING | pattern 'official_sms_check.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.